In [3]:
# Cargamos Librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# modelos lineales
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# indicadores de ajuste del modelo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,mean_squared_error,r2_score,confusion_matrix,recall_score,classification_report,precision_score,f1_score
from sklearn.metrics import roc_curve, roc_auc_score

# modelos de árboles de decisión
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import GridSearchCV

In [4]:
# cargamos datos

health_df = pd.read_csv('evaluacionh2/healthcare_dataset.csv',sep=',', encoding='iso-8859-1')

print(health_df.shape) #
health_df.head(2)

(55500, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive


In [5]:
health_df['Blood Type'].value_counts()

Blood Type
A-     6969
A+     6956
AB+    6947
AB-    6945
B+     6945
B-     6944
O+     6917
O-     6877
Name: count, dtype: int64

In [6]:
health_df.columns

Index(['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition',
       'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider',
       'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date',
       'Medication', 'Test Results'],
      dtype='str')

In [7]:
# una copia a los datos
df = health_df.copy()

In [8]:
# convertir a dummies las variables categoricas o cualitativas
df = pd.get_dummies(df, columns=['Blood Type','Medical Condition'], drop_first=True)
print(df.shape)
df.head(2)

(55500, 25)


,Name,Age,Gender,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,...,Blood Type_AB-,Blood Type_B+,Blood Type_B-,Blood Type_O+,Blood Type_O-,Medical Condition_Asthma,Medical Condition_Cancer,Medical Condition_Diabetes,Medical Condition_Hypertension,Medical Condition_Obesity
0,Bobby JacksOn,30,Male,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,...,False,False,True,False,False,False,True,False,False,False
1,LesLie TErRy,62,Male,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,...,False,False,False,False,False,False,False,False,False,True


In [9]:
## convertir las variables booleanas a enteros
#df = df.replace({True: 1, False: 0})
#df.head(2)

for col in df.columns:
    if col.startswith('Blood Type') or col.startswith('Medical Condition'):
        if df[col].dtype == bool:
            df[col] = df[col].astype(int)
df[[c for c in df.columns if c.startswith('Blood Type') or c.startswith('Medical Condition')]].head(2)

print(df.shape)
df.head(2)

(55500, 25)


,Name,Age,Gender,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,...,Blood Type_AB-,Blood Type_B+,Blood Type_B-,Blood Type_O+,Blood Type_O-,Medical Condition_Asthma,Medical Condition_Cancer,Medical Condition_Diabetes,Medical Condition_Hypertension,Medical Condition_Obesity
0,Bobby JacksOn,30,Male,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,...,0,0,1,0,0,0,1,0,0,0
1,LesLie TErRy,62,Male,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,...,0,0,0,0,0,0,0,0,0,1


In [10]:
df.columns

Index(['Name', 'Age', 'Gender', 'Date of Admission', 'Doctor', 'Hospital',
       'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type',
       'Discharge Date', 'Medication', 'Test Results', 'Blood Type_A-',
       'Blood Type_AB+', 'Blood Type_AB-', 'Blood Type_B+', 'Blood Type_B-',
       'Blood Type_O+', 'Blood Type_O-', 'Medical Condition_Asthma',
       'Medical Condition_Cancer', 'Medical Condition_Diabetes',
       'Medical Condition_Hypertension', 'Medical Condition_Obesity'],
      dtype='str')

In [18]:
health_df ['Medical Condition']= health_df ['Medical Condition'].str[0]
##health_df ['Medical Condition']= health_df ['Medical Condition'].astype[int]


In [21]:
health_df ['Medical Condition']= health_df ['Medical Condition'].astype('category').cat.codes

In [23]:
health_df.dtypes

Name                      str
Age                     int64
Gender                    str
Blood Type                str
Medical Condition        int8
Date of Admission         str
Doctor                    str
Hospital                  str
Insurance Provider        str
Billing Amount        float64
Room Number             int64
Admission Type            str
Discharge Date            str
Medication                str
Test Results              str
dtype: object

In [11]:
df[['Age','Gender','Billing Amount','Admission Type']].dtypes

Age                 int64
Gender                str
Billing Amount    float64
Admission Type        str
dtype: object

In [25]:
# seleccionamos variables explicativas y variables dependiente
#X = df[['Kidhome','Recency','education_Basic','Complain','Income','MntTotal','Age']]
X = df[['Age','Room Number']] # variables independientes
y = health_df['Medical Condition'] # variable dependiente o variable a predecir o variable objetivo

In [26]:
health_df.dtypes

Name                      str
Age                     int64
Gender                    str
Blood Type                str
Medical Condition        int8
Date of Admission         str
Doctor                    str
Hospital                  str
Insurance Provider        str
Billing Amount        float64
Room Number             int64
Admission Type            str
Discharge Date            str
Medication                str
Test Results              str
dtype: object

In [27]:
y.head(2)

0    1
1    4
Name: Medical Condition, dtype: int8

In [28]:
# Dividir los datos en conjuntos de entrenamiento y prueba o testeo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777)

In [30]:
print(X_train_numeric.dtypes)

NameError: name 'X_train_numeric' is not defined

In [44]:
print(y_train.head())

49386    Emergency
48037    Emergency
48587    Emergency
23365       Urgent
7005      Elective
Name: Admission Type, dtype: str


In [47]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)   # convierte texto → números

In [48]:
# Dividir los datos en conjuntos de entrenamiento y prueba o testeo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777)

In [51]:
# =========================================
# 1. Convertir variables categóricas de X
# =========================================

import pandas as pd

X = pd.get_dummies(X, drop_first=True)  # convierte 'Male', 'Female', etc. en columnas 0/1


# =========================================
# 2. Codificar y (etiquetas)
# =========================================
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)


# =========================================
# 3. Dividir datos
# =========================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=777
)


# =========================================
# 4. Entrenar modelo con GridSearchCV
# =========================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

clf = DecisionTreeClassifier()

param_grid = {
    'max_depth': list(range(1, 20)),
    'min_samples_split': list(range(2, 10)),
    'min_samples_leaf': list(range(1, 4))
}

grid_search = GridSearchCV(clf, param_grid, cv=5)
grid_search.fit(X_train, y_train)


# =========================================
# 5. Ver mejores hiperparámetros
# =========================================
best_params = grid_search.best_params_
best_params

{'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 2}